# Evaluation (Qwen3.5-4B): PA-SFT model on a single benchmark

## 1. Install dependencies

In [ ]:
!pip install -q \
  "vllm==0.17.0" \
  "huggingface_hub>=0.30,<0.36" \
  pillow

!pip install -q --no-deps --upgrade "tokenizers>=0.22"
!pip install -q --no-deps "transformers==5.3.0"

!pip install -q --no-deps --force-reinstall "huggingface_hub>=1.3.0,<2.0"
!pip install -q --no-deps --force-reinstall "tokenizers==0.22.2"

import importlib.metadata as md
for pkg in ["vllm", "transformers", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:18s} NOT INSTALLED")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. HuggingFace login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN   = userdata.get('HF_TOKEN')
HF_REPO_ID = 'minsu0567/IAD-X1-GRPO-answer-last-no-hard'
login(token=HF_TOKEN)
print('HuggingFace login OK.')

## 4. GPU check

In [ ]:
!nvidia-smi

## 5. Path sanity check

In [ ]:
import os

DRIVE_ROOT    = '/content/drive/MyDrive'
IAD_R1_DIR    = f'{DRIVE_ROOT}/IAD-R1-main'
IAD_X1_DIR    = f'{DRIVE_ROOT}/IAD-X1'
EVAL_SRC      = f'{IAD_X1_DIR}/eval_src'
EVAL_DATA_DIR = f'{DRIVE_ROOT}/Uni-IAD_eval_dataset'
OUTPUT_DIR    = '/content/eval_results'

for p in [IAD_R1_DIR, EVAL_SRC, EVAL_DATA_DIR]:
    assert os.path.isdir(p), f'Missing directory: {p}'
assert os.path.isfile(f'{EVAL_SRC}/eval_benchmark_qwen3_5.py'), 'eval_benchmark_qwen3_5.py not found'
assert os.path.isfile(f'{IAD_R1_DIR}/helper/summary.py'), 'IAD-R1 helper/summary.py not found'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('All paths OK.')

## 6. Choose benchmark and run evaluation

In [ ]:
import sys

if EVAL_SRC not in sys.path:
    sys.path.append(EVAL_SRC)

sys.argv = ['eval_benchmark_qwen3_5.py']
import eval_benchmark_qwen3_5 as ev

ev.GENERAL_QUESTION_PROMPT = (
    'You are an expert in detecting industrial anomalies in images. '
    'You will be provided with two images, a reference image (ref_img) as a normal standard and a query image (query_img) for inspection. '
    'Your reasoning and response must strictly follow these constraints based on the specific tags. '
    '\n1. Analyze only the ref_img and define the characteristics of a normal state such as structural integrity, surface texture, and component completeness. '
    '2. Inspect the query_img for any anomalies by comparing it against the ref_img baseline. '
    '\nIf you find anomalies in the query image, respond with <type>...</type><location>...</location><answer>Yes</answer>'
    '\nIf no anomalies are detected in the query image, respond with <answer>No</answer> '
)
print('Overrode eval_benchmark_qwen3_5.GENERAL_QUESTION_PROMPT.')

if not hasattr(ev, '_build_prompt_orig'):
    ev._build_prompt_orig = ev.build_prompt
_orig_build_prompt = ev._build_prompt_orig

def _build_prompt_no_think(processor):
    prompt = _orig_build_prompt(processor)
    prompt = prompt.rstrip()
    if prompt.endswith('<think>'):
        prompt = prompt[: -len('<think>')]
    return prompt
ev.build_prompt = _build_prompt_no_think
print('Patched eval_benchmark_qwen3_5.build_prompt -> <think> primer stripped.')

BENCHMARK     = 'DAGM'
BATCH_SIZE    = 4
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL  = 0.9

sys.argv = [
    'eval_benchmark_qwen3_5.py',
    '--benchmark', BENCHMARK,
    '--model_path', HF_REPO_ID,
    '--data_root', EVAL_DATA_DIR,
    '--output_dir', OUTPUT_DIR,
    '--iad_r1_root', IAD_R1_DIR,
    '--batch_size', str(BATCH_SIZE),
    '--max_model_len', str(MAX_MODEL_LEN),
    '--gpu_memory_utilization', str(GPU_MEM_UTIL),
]
ev.main()